# 02 — Silver: limpeza, tipagem e normalização

**Regras:** valores não recuperáveis ficam NULL; não inventamos notas, dinheiro ou nomes para substituir dados corrompidos. Registros sem um ID numérico válido não podem identificar um filme e são excluídos das tabelas de negócio. Dados originais continuam disponíveis na Bronze e nos CSVs do Volume.

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import re

hoje = datetime.now(ZoneInfo("America/Sao_Paulo")).date()
def parametro(nome, padrao):
    try:
        return dbutils.widgets.get(nome)
    except Exception:
        dbutils.widgets.text(nome, padrao)
        return dbutils.widgets.get(nome)

CATALOGO = parametro("catalogo", "workspace")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOGO):
    raise ValueError("Use um nome simples de catálogo, com letras, números e underscore.")
DATA_INICIO = datetime.strptime(parametro("data_inicio", (hoje-timedelta(days=6)).strftime("%m-%d-%Y")), "%m-%d-%Y").date()
DATA_FIM = datetime.strptime(parametro("data_fim", hoje.strftime("%m-%d-%Y")), "%m-%d-%Y").date()
if DATA_INICIO > DATA_FIM or DATA_FIM > hoje:
    raise ValueError("O período deve estar em ordem e não pode terminar no futuro.")

def tabela(camada, nome):
    return f"{CATALOGO}.{camada}.{nome}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.silver")

def gravar_silver(df, nome):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(tabela("silver", nome))

def identificador(c):
    x = F.trim(c.cast("string"))
    return F.when(x.rlike(r"^[0-9]+$"), x)

def latest(df, chave="id"):
    colunas = sorted(c for c in df.columns if c != "ingestion_datetime")
    preenchimento = sum((F.when(F.length(F.trim(F.col(c))) > 0, 1).otherwise(0) for c in colunas), F.lit(0))
    desempate = F.sha2(F.to_json(F.struct(*[F.col(c) for c in colunas])), 256)
    janela = Window.partitionBy(chave).orderBy(
        F.col("ingestion_datetime").desc_nulls_last(), preenchimento.desc(), desempate.desc()
    )
    return (df.withColumn("_versao", F.row_number().over(janela))
        .filter(F.col("_versao") == 1).drop("_versao"))

def cast_seguro(df, origem, destino, tipo):
    return df.withColumn(destino, F.expr(f"try_cast(`{origem}` AS {tipo})"))

def decimal_normalizado(c, moeda=False):
    x = F.upper(F.trim(c.cast("string")))
    if moeda:
        x = F.regexp_replace(x, r"(?i)(USD|US\$|R\$|BRL|\$|€|£)", "")
    x = F.regexp_replace(x, r"\s+", "")
    ambos = x.contains(",") & x.contains(".")
    virgula_final = x.rlike(r",[0-9]+$")
    com_ambos = F.when(virgula_final,
        F.regexp_replace(F.regexp_replace(x, r"\.", ""), ",", ".")
    ).otherwise(F.regexp_replace(x, ",", ""))
    if moeda:
        milhares = x.rlike(r"^[+-]?[0-9]{1,3}([.,][0-9]{3})+$")
        simples = F.when(milhares, F.regexp_replace(x, r"[.,]", "")).otherwise(F.regexp_replace(x, ",", "."))
    else:
        simples = F.regexp_replace(x, ",", ".")
    resultado = F.when(ambos, com_ambos).otherwise(simples)
    return F.when(resultado.rlike(r"^[+-]?[0-9]+(?:\.[0-9]+)?$"), resultado)

## 1. Histórico contínuo de cotações

In [ ]:
raw_fx = spark.table(tabela("bronze", "tb_cotacao_dolar"))
fx = raw_fx.select(
    F.expr("try_cast(dataHoraCotacao AS TIMESTAMP)").alias("data_hora_cotacao"),
    F.expr("try_cast(cotacaoCompra AS DECIMAL(18,6))").alias("cotacao_compra"),
    F.col("ingestion_datetime").alias("data_ingestao")
).filter(F.col("data_hora_cotacao").isNotNull() & (F.col("cotacao_compra") > 0))
fx = fx.withColumn("data_cotacao", F.to_date("data_hora_cotacao"))
# Descarta cotações fora do intervalo de apoio desta execução.
fx = fx.filter(F.col("data_cotacao").between(F.lit(DATA_INICIO-timedelta(days=15)), F.lit(DATA_FIM)))
w_fx = Window.partitionBy("data_cotacao").orderBy(F.col("data_hora_cotacao").desc(), F.col("data_ingestao").desc())
diarias = fx.withColumn("_n", F.row_number().over(w_fx)).filter("_n = 1").drop("_n")
primeira = diarias.agg(F.min("data_cotacao").alias("inicio")).first()["inicio"]
if primeira is None or primeira > DATA_INICIO:
    raise RuntimeError("Falta uma cotação anterior ou igual ao início. Amplie o apoio da API na Bronze.")
calendario = spark.range(1).select(F.explode(F.sequence(F.lit(primeira), F.lit(DATA_FIM))).alias("data_cotacao"))
w_fill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)
historico = (calendario.join(diarias, "data_cotacao", "left")
    .withColumn("data_cotacao_origem", F.when(F.col("cotacao_compra").isNotNull(), F.col("data_cotacao")))
    .withColumn("cotacao_compra", F.last("cotacao_compra", ignorenulls=True).over(w_fill))
    .withColumn("data_cotacao_origem", F.last("data_cotacao_origem", ignorenulls=True).over(w_fill))
    .withColumn("data_hora_cotacao", F.last("data_hora_cotacao", ignorenulls=True).over(w_fill))
    .withColumn("data_ingestao", F.last("data_ingestao", ignorenulls=True).over(w_fill))
    .filter(F.col("data_cotacao").between(F.lit(DATA_INICIO), F.lit(DATA_FIM))))
if historico.filter(F.col("cotacao_compra").isNull()).limit(1).count():
    raise RuntimeError("A série de cotação ainda contém dias sem taxa.")
gravar_silver(historico, "tb_cotacao_dolar")
display(historico.orderBy("data_cotacao"))


## 2. Informações dos filmes

Os formatos encontrados são `yyyy-MM-dd`, `dd/MM/yyyy` e `MM-dd-yyyy`. Utilizamos `try_to_timestamp` e `coalesce` para tentar esses padrões sem interromper o pipeline. O padrão com hífen e ano ao final prioriza **mês-dia-ano**, como no arquivo; `dd-MM-yyyy` é uma tentativa adicional apenas quando a anterior falha. Datas ambíguas não permitem deduzir a intenção sem uma regra explícita.

O status é convertido para minúsculas; hífens, underscores, pontuação e espaços extras são normalizados antes do mapeamento. Valores fora do dicionário recebem **Não Informado**. A duração deve ser um inteiro positivo.


In [ ]:
info_raw = spark.table(tabela("bronze", "tb_movies_info"))
info_raw = info_raw.withColumn("id", identificador(F.col("id"))).filter(F.col("id").isNotNull())
info_raw = latest(info_raw)
data_texto = F.trim(F.col("release_date"))
data_convertida = F.coalesce(*[
    F.to_date(F.try_to_timestamp(data_texto, F.lit(p)))
    for p in ["yyyy-MM-dd", "dd/MM/yyyy", "MM-dd-yyyy", "dd-MM-yyyy", "yyyy/MM/dd", "yyyy-MM-dd HH:mm:ss"]
])
status_normalizado = F.trim(F.regexp_replace(
    F.regexp_replace(F.lower(F.trim(F.col("status"))), r"[^a-z\s]", " "), r"\s+", " "
))
traducao = F.create_map(*[F.lit(v) for par in {
    "released": "Lançado", "post production": "Pós-Produção", "in production": "Em Produção",
    "planned": "Planejado", "rumored": "Rumores", "canceled": "Cancelado", "cancelled": "Cancelado"
}.items() for v in par])
info = info_raw.select(
    F.col("id").alias("id_filme"), F.trim("title").alias("titulo"),
    F.trim("original_title").alias("titulo_original"), data_convertida.alias("data_lancamento"),
    F.expr("try_cast(runtime AS INT)").alias("duracao_minutos"),
    F.lower(F.trim("original_language")).alias("idioma_original"),
    F.coalesce(traducao[status_normalizado], F.lit("Não Informado")).alias("status_filme"),
    F.col("overview").alias("sinopse"), F.col("tagline").alias("frase_divulgacao"),
    F.col("ingestion_datetime").alias("data_ingestao")
).withColumn("ano_lancamento", F.year("data_lancamento"))
info = info.withColumn("duracao_minutos", F.when(F.col("duracao_minutos") > 0, F.col("duracao_minutos")))
gravar_silver(info, "tb_info_filmes")
display(info.limit(10))


## 3. Financeiro

Somente símbolos monetários conhecidos são removidos, espaços e separadores de milhar. Textos como `Unknown`, `N/A` e `Não Informado` não passam pela validação numérica e ficam NULL. Os sufixos também são interpretados **K, M e B** como mil, milhão e bilhão; por exemplo, `34.0M` equivale a 34 milhões.

Valores USD são DECIMAL(18,2); valores menores ou iguais a zero são ausentes. Para comparar os filmes em uma mesma base monetária, a **cotação mais recente do período escolhido** é aplicada, e não uma cotação histórica da estreia. A taxa e sua data para rastreabilidade também são guardadas.

Lucro = receita − orçamento. Margem percentual = lucro / receita × 100. Se faltar receita ou orçamento, lucro e margem ficam NULL: ausência de dado não é interpretada como zero. Lucro negativo é válido.


In [ ]:
financeiro_raw = spark.table(tabela("bronze", "tb_movies_financials"))
financeiro_raw = latest(financeiro_raw.withColumn("id", identificador(F.col("id"))).filter(F.col("id").isNotNull()))
financeiro = financeiro_raw.select(F.col("id").alias("id_filme"), "budget", "revenue", F.col("ingestion_datetime").alias("data_ingestao"))
for origem, destino in [("budget", "orcamento_usd"), ("revenue", "receita_usd")]:
    texto = F.upper(F.trim(F.col(origem)))
    sufixo = F.regexp_extract(texto, r"([KMB])$", 1)
    multiplicador = (F.when(sufixo == "K", 1000).when(sufixo == "M", 1000000)
        .when(sufixo == "B", 1000000000).otherwise(1))
    sem_sufixo = F.regexp_replace(texto, r"[KMB]$", "")
    # Com K/M/B, o número anterior ao sufixo é uma mantissa decimal.
    mantissa = F.regexp_replace(sem_sufixo, r"(?i)(USD|US\$|R\$|BRL|\$|€|£)", "")
    valor_texto = F.when(sufixo != "", decimal_normalizado(mantissa)).otherwise(decimal_normalizado(sem_sufixo, moeda=True))
    financeiro = financeiro.withColumn("_valor_texto", valor_texto)
    financeiro = cast_seguro(financeiro, "_valor_texto", "_valor", "DECIMAL(24,6)")
    financeiro = financeiro.withColumn("_total", F.col("_valor") * multiplicador)
    financeiro = cast_seguro(financeiro, "_total", destino, "DECIMAL(18,2)")
    financeiro = financeiro.withColumn(destino, F.when(F.col(destino) > 0, F.col(destino)))

ultima_taxa = (spark.table(tabela("silver", "tb_cotacao_dolar"))
    .orderBy(F.col("data_cotacao").desc()).limit(1)
    .select(F.col("cotacao_compra").alias("cotacao_usada"), F.col("data_cotacao_origem").alias("data_cotacao_usada")))
financeiro = financeiro.crossJoin(ultima_taxa)
financeiro = (financeiro
    .withColumn("_orc_brl", F.col("orcamento_usd") * F.col("cotacao_usada"))
    .withColumn("_rec_brl", F.col("receita_usd") * F.col("cotacao_usada"))
    .withColumn("_luc_usd", F.col("receita_usd") - F.col("orcamento_usd"))
    .withColumn("_luc_brl", F.col("_luc_usd") * F.col("cotacao_usada")))
for origem, destino in [("_orc_brl", "orcamento_brl"), ("_rec_brl", "receita_brl"), ("_luc_usd", "lucro_usd"), ("_luc_brl", "lucro_brl")]:
    financeiro = cast_seguro(financeiro, origem, destino, "DECIMAL(18,2)")
financeiro = financeiro.withColumn("margem_lucro_percentual", F.round(
    F.when(F.col("receita_usd") > 0, F.col("lucro_usd") / F.col("receita_usd") * 100), 2
).cast("decimal(18,2)"))
financeiro = financeiro.select("id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
    "lucro_usd", "lucro_brl", "margem_lucro_percentual", "cotacao_usada", "data_cotacao_usada", "data_ingestao")
gravar_silver(financeiro, "tb_financeiro_filmes")
display(financeiro.limit(10))


## 4. Métricas de engajamento

Popularidade aceita ponto ou vírgula decimal, sem confundir três casas decimais com milhar. `try_cast` converte apenas valores compatíveis. Notas fora de 0–10 viram NULL; valores multiplicados por erro de escala não são corrigidos por suposição. Votos aceitam apenas inteiros não negativos e dentro do limite INT, evitando truncar valores como `7.5` para `7`.


In [ ]:
metricas_raw = spark.table(tabela("bronze", "tb_movies_metrics"))
metricas_raw = latest(metricas_raw.withColumn("id", identificador(F.col("id"))).filter(F.col("id").isNotNull()))
metricas = metricas_raw.select(F.col("id").alias("id_filme"), "popularity", "vote_average", "vote_count", "averageRating", "numVotes", F.col("ingestion_datetime").alias("data_ingestao"))
for origem, destino in [("popularity", "popularidade"), ("vote_average", "nota_media_tmdb"), ("averageRating", "nota_media_imdb")]:
    metricas = metricas.withColumn("_numero", decimal_normalizado(F.col(origem)))
    metricas = cast_seguro(metricas, "_numero", destino, "DOUBLE")
    valida = F.col(destino) >= 0 if destino == "popularidade" else F.col(destino).between(0, 10)
    metricas = metricas.withColumn(destino, F.when(valida, F.col(destino)))
for origem, destino in [("vote_count", "qtd_votos_tmdb"), ("numVotes", "qtd_votos_imdb")]:
    metricas = metricas.withColumn("_inteiro", F.when(F.trim(F.col(origem)).rlike(r"^\+?[0-9]+$"), F.trim(F.col(origem))))
    metricas = cast_seguro(metricas, "_inteiro", destino, "INT")
    metricas = metricas.withColumn(destino, F.when(F.col(destino) >= 0, F.col(destino)))
metricas = metricas.select("id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb", "data_ingestao")
gravar_silver(metricas, "tb_metricas_engajamento")
display(metricas.limit(10))


## 5. Avaliações de usuários

Duplicatas são removidas pela combinação filme, usuário, nota e comentário. Notas inválidas viram NULL e comentários vazios recebem **Sem comentário**. Uma segunda deduplicação após a padronização evita duplicatas que só diferiam por espaços ou pela representação numérica da nota.


In [ ]:
reviews_raw = spark.table(tabela("bronze", "tb_movies_reviews"))
reviews_raw = (reviews_raw.withColumn("id", identificador(F.col("id")))
    .filter(F.col("id").isNotNull())
    .groupBy("id", "nome", "nota", "comentario").agg(F.max("ingestion_datetime").alias("ingestion_datetime")))
reviews = reviews_raw.select(
    F.col("id").alias("id_filme"), F.trim("nome").alias("nome_usuario"),
    decimal_normalizado(F.col("nota")).alias("_nota"),
    F.when(F.col("comentario").isNull() | (F.length(F.trim("comentario")) == 0), "Sem comentário")
        .otherwise(F.trim("comentario")).alias("comentario_usuario"),
    F.col("ingestion_datetime").alias("data_ingestao"))
reviews = cast_seguro(reviews, "_nota", "nota_usuario", "DOUBLE")
reviews = reviews.withColumn("nota_usuario", F.when(F.col("nota_usuario").between(0, 10), F.col("nota_usuario")))
reviews = (reviews.groupBy("id_filme", "nome_usuario", "nota_usuario", "comentario_usuario")
    .agg(F.max("data_ingestao").alias("data_ingestao")))
gravar_silver(reviews, "tb_avaliacoes_usuarios")
display(reviews.limit(10))


## 6. Gêneros, pessoas e empresas

As vírgulas, ponto e vírgula **e barras verticais (`|`)** são padronizadas antes de `split` + `explode`. Os gêneros são validados contra o domínio presente na base TMDB. Nomes de entidade recebem `initcap`, espaços padronizados e deduplicação por filme, nome e tipo.

In [ ]:
tags = spark.table(tabela("bronze", "tb_credits_and_tags"))
tags = latest(tags.withColumn("id", identificador(F.col("id"))).filter(F.col("id").isNotNull()))
GENEROS = ["Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama", "Family", "Fantasy",
    "History", "Horror", "Music", "Mystery", "Romance", "Science Fiction", "TV Movie", "Thriller", "War", "Western"]
DOMINIO_GENEROS = [g.lower() for g in GENEROS]

def elementos(c):
    return F.split(F.regexp_replace(c, r"[;|]", ","), ",")
generos = (tags.select(F.col("id").alias("id_filme"), F.explode(elementos(F.col("genres"))).alias("_genero"))
    .withColumn("_genero", F.lower(F.trim("_genero")))
    .filter(F.col("_genero").isin(DOMINIO_GENEROS)))
mapa_generos = F.create_map(*[F.lit(v) for g in GENEROS for v in (g.lower(), g)])
generos = generos.select("id_filme", mapa_generos[F.col("_genero")].alias("nome_genero")).dropDuplicates()
gravar_silver(generos, "tb_generos")

RUIDOS = DOMINIO_GENEROS + [
    "english", "french", "german", "spanish", "italian", "japanese", "korean", "portuguese", "russian",
    "mandarin", "cantonese", "hindi", "arabic", "polish", "swedish", "dutch", "turkish", "no language",
    "united states of america", "united kingdom", "france", "germany", "italy", "spain", "japan", "china",
    "canada", "brazil", "india", "south korea", "russia", "australia", "mexico", "argentina",
    "unknown", "não informado", "nenhum", "none", "null", "n/a", "nan", "[]"
]

partes_genero = F.transform(elementos(F.coalesce(F.col("genres"), F.lit(""))), lambda x: F.lower(F.trim(x)))
validade_tags = F.forall(partes_genero, lambda x: (x == "") | x.isin(DOMINIO_GENEROS))
tags_entidades = tags.filter(validade_tags)
partes = []

for origem, tipo in [("cast", "Ator"), ("directors", "Diretor"), ("writers", "Roteirista"), ("production_companies", "Produtora")]:
    entidade = tags_entidades.select(F.col("id").alias("id_filme"),
        F.explode(elementos(F.col(origem))).alias("_nome"), F.lit(tipo).alias("tipo_entidade"))
    entidade = entidade.withColumn("_nome", F.trim(F.regexp_replace("_nome", r"\s+", " ")))
    # Pessoas não recebem dígitos; produtoras podem ter números, ex.: 20th Century Fox.
    letras = F.col("_nome").rlike(r"\p{L}")
    estrutura = F.col("_nome").rlike(r"^[\p{L}\p{M}0-9 .&'’()\-+!]+$")
    sem_descricao = ~F.lower(F.col("_nome")).rlike(r"^(who |which |when |while |where |and |but |he |she |they |his |her |their |after |before |this |it )")
    tamanho = F.length("_nome").between(1, 100) & (F.size(F.split("_nome", " ")) <= 12)
    dominio = ~F.lower(F.col("_nome")).isin(RUIDOS)
    nome_pessoa = (F.col("tipo_entidade") == "Produtora") | ~F.col("_nome").rlike(r"[0-9]")
    entidade = entidade.filter(letras & estrutura & tamanho & dominio & nome_pessoa & sem_descricao)
    partes.append(entidade.select("id_filme", F.initcap("_nome").alias("nome_entidade"), "tipo_entidade"))
pessoas_empresas = partes[0]

for parte in partes[1:]:
    pessoas_empresas = pessoas_empresas.unionByName(parte)
pessoas_empresas = pessoas_empresas.dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])

gravar_silver(pessoas_empresas, "tb_pessoas_empresas")
display(generos.limit(10))
display(pessoas_empresas.limit(10))


## 7. Validações da camada

Verificamos unicidade e as faixas numéricas, em vez de apenas conferir se o notebook executou. O quadro de qualidade também mostra quantos registros ficaram sem uma informação crítica.


In [ ]:
for nome in ["tb_info_filmes", "tb_financeiro_filmes", "tb_metricas_engajamento"]:
    df = spark.table(tabela("silver", nome))
    if df.groupBy("id_filme").count().filter("count > 1").limit(1).count():
        raise AssertionError(f"Há filmes duplicados em {nome}.")
metricas = spark.table(tabela("silver", "tb_metricas_engajamento"))

for c in ["nota_media_tmdb", "nota_media_imdb"]:
    assert metricas.filter(F.col(c).isNotNull() & ~F.col(c).between(0, 10)).limit(1).count() == 0
qualidade = spark.table(tabela("silver", "tb_info_filmes")).agg(
    F.count("*").alias("qtd_filmes"),
    F.sum(F.col("data_lancamento").isNull().cast("int")).alias("qtd_sem_data"),
    F.sum((F.col("status_filme") == "Não Informado").cast("int")).alias("qtd_status_nao_informado"))

display(qualidade)
print("Validações Silver concluídas.")
